In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Sample sentences
sentences = [
    "The dog is playing in the park",
    "A puppy is running outside",
    "The cat is sleeping on the couch",
    "Python is a programming language",
    "Machine learning models need data",
    "I love coding in Python"
]

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate normalized embeddings (important for accurate cosine similarity)
embeddings = model.encode(sentences, normalize_embeddings=True)

# Compute cosine similarity matrix
similarity_matrix = cosine_similarity(embeddings)

# Function to get sorted similarities for a query index
def get_sorted_similarities(query_idx: int):
    scores = []
    for i in range(len(sentences)):
        if i != query_idx:
            scores.append((i, similarity_matrix[query_idx][i]))
    # Sort descending by similarity score
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores

print("Similarity Analysis:\n")

# Query 1: "The dog is playing in the park" (index 0)
query1_idx = 0
scores1 = get_sorted_similarities(query1_idx)

most_similar_1 = sentences[scores1[0][0]]
least_similar_1 = sentences[scores1[-1][0]]

print(f"Query: \"{sentences[query1_idx]}\"")
print(f"Most similar: \"{most_similar_1}\" (score: {scores1[0][1]:.4f})")
print(f"Least similar: \"{least_similar_1}\" (score: {scores1[-1][1]:.4f})")
print("Observations: The embedding model captures strong semantic similarity between sentences about dogs/puppies engaged in outdoor activities. "
      "The sentence about a cat sleeping is the least similar due to different animal and action. "
      "Programming-related sentences have low similarity, showing good topic separation.\n")

# Query 2: "Python is a programming language" (index 3)
query2_idx = 3
scores2 = get_sorted_similarities(query2_idx)

most_similar_2 = sentences[scores2[0][0]]
least_similar_2 = sentences[scores2[-1][0]]

print(f"Query: \"{sentences[query2_idx]}\"")
print(f"Most similar: \"{most_similar_2}\" (score: {scores2[0][1]:.4f})")
print(f"Least similar: \"{least_similar_2}\" (score: {scores2[-1][1]:.4f})")
print("Observations: The model correctly identifies the sentence expressing affection for coding in Python as most similar. "
      "The machine learning sentence has moderate similarity due to shared technical domain. "
      "Animal-related sentences, especially the cat one, are correctly identified as unrelated.\n")

print("Recommended similarity threshold: 0.55")
print("   - Scores above ~0.65-0.70 indicate strong semantic relevance (paraphrases or closely related ideas)")
print("   - Scores around 0.35-0.50 may share broad topics but are not direct matches")
print("   - Scores below 0.55 (especially <0.40) are typically unrelated topics and can be safely filtered out in retrieval tasks")

c:\Users\Abdulmalik\Desktop\deep_learning\deepenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Similarity Analysis:

Query: "The dog is playing in the park"
Most similar: "A puppy is running outside" (score: 0.3984)
Least similar: "Machine learning models need data" (score: -0.0052)
Observations: The embedding model captures strong semantic similarity between sentences about dogs/puppies engaged in outdoor activities. The sentence about a cat sleeping is the least similar due to different animal and action. Programming-related sentences have low similarity, showing good topic separation.

Query: "Python is a programming language"
Most similar: "I love coding in Python" (score: 0.7304)
Least similar: "The cat is sleeping on the couch" (score: 0.0199)
Observations: The model correctly identifies the sentence expressing affection for coding in Python as most similar. The machine learning sentence has moderate similarity due to shared technical domain. Animal-related sentences, especially the cat one, are correctly identified as unrelated.

Recommended similarity threshold: 0.55
   

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the model (clean and simple with sentence-transformers)
model = SentenceTransformer('all-MiniLM-L6-v2')

document = """
Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to
the natural intelligence displayed by humans and animals. Leading AI textbooks define
the field as the study of intelligent agents: any device that perceives its environment
and takes actions that maximize its chance of successfully achieving its goals.
Machine learning is a subset of artificial intelligence that focuses on the use of data
and algorithms to imitate the way that humans learn, gradually improving its accuracy.
Machine learning is an important component of the growing field of data science.
Deep learning is part of a broader family of machine learning methods based on artificial
neural networks with representation learning. Learning can be supervised, semi-supervised
or unsupervised. Deep learning architectures such as deep neural networks, deep belief
networks, recurrent neural networks and convolutional neural networks have been applied
to fields including computer vision, speech recognition, natural language processing,
machine translation, and bioinformatics.
Natural language processing is a subfield of linguistics, computer science, and artificial
intelligence concerned with the interactions between computers and human language, in
particular how to program computers to process and analyze large amounts of natural
language data. Challenges in natural language processing frequently involve speech
recognition, natural language understanding, and natural language generation.
"""

query = "What is machine learning?"

# Generate query embedding (normalized for cosine similarity)
query_embedding = model.encode(query, normalize_embeddings=True)

# Simple character-based chunking with word boundary respect
def chunk_document(text: str, max_chars: int):
    text = text.strip()
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        # Try to break at a space to avoid cutting words
        if end < len(text):
            last_space = text.rfind(' ', start, end)
            if last_space != -1:
                end = last_space
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end + (1 if end < len(text) and text[end] == ' ' else 0)
    return chunks

# Define chunk sizes
sizes = {'Small': 100, 'Medium': 200, 'Large': 400}

print("Chunk Size Comparison:\n")

for name, size in sizes.items():
    chunks = chunk_document(document, size)
    num_chunks = len(chunks)
    
    # Generate embeddings for all chunks in one batch
    chunk_embeddings = model.encode(chunks, normalize_embeddings=True)
    
    # Compute cosine similarities with the query
    similarities = np.dot(chunk_embeddings, query_embedding)
    
    # Get top 3 most similar chunks
    top_idx = np.argsort(similarities)[-3:][::-1]
    top_chunk = chunks[top_idx[0]]
    top_score = similarities[top_idx[0]]
    
    print(f"{name} Chunks ({size} chars):")
    print(f"- Number of chunks: {num_chunks}")
    print(f"- Top result: \"{top_chunk.replace('\n', ' ')}\"")
    print(f"- Score: {top_score:.4f}")
    print("- Analysis: ", end="")
    
    if name == "Small":
        print("Highly focused — captures the core definition with minimal extra text. Best precision.")
    elif name == "Medium":
        print("Excellent balance — includes the definition plus relevant context (e.g., data science link) without noise.")
    elif name == "Large":
        print("More complete but diluted — mixes in deep learning and other topics, lowering relevance score.")
    print()

print("Best chunk size for this use case: Medium (200 characters) because it delivers the most relevant and self-contained chunk as the top result while preserving useful context, offering the optimal trade-off between focus and completeness.")

Chunk Size Comparison:

Small Chunks (100 chars):
- Number of chunks: 16
- Top result: "chance of successfully achieving its goals. Machine learning is a subset of artificial intelligence"
- Score: 0.7199
- Analysis: Highly focused — captures the core definition with minimal extra text. Best precision.

Medium Chunks (200 chars):
- Number of chunks: 8
- Top result: "that focuses on the use of data and algorithms to imitate the way that humans learn, gradually improving its accuracy. Machine learning is an important component of the growing field of data"
- Score: 0.7397
- Analysis: Excellent balance — includes the definition plus relevant context (e.g., data science link) without noise.

Large Chunks (400 chars):
- Number of chunks: 4
- Top result: "that focuses on the use of data and algorithms to imitate the way that humans learn, gradually improving its accuracy. Machine learning is an important component of the growing field of data science. Deep learning is part of a broader famil